# Cek anomali harga produk
Membaca `lakehouse.distribusi.harga_produk` lewat query engine platform (governance sama dengan SQL Workspace), lalu membandingkannya dengan snapshot sebelumnya.

In [1]:
import os, pandas as pd, trino.auth
from sqlalchemy import create_engine
engine = create_engine(
    f"trino://{os.environ['TRINO_USER']}@{os.environ['TRINO_HOST']}:{os.environ['TRINO_PORT']}",
    connect_args={"http_scheme": "https",
                  "auth": trino.auth.JWTAuthentication(os.environ["TRINO_JWT_TOKEN"]),
                  "verify": os.environ["TRINO_TLS_CA_CERT"]},
)
df = pd.read_sql("SELECT * FROM lakehouse.distribusi.harga_produk", engine)
df["harga_rupiah"] = df["harga_rupiah"].astype(float)
print(len(df), "produk")

1200 produk


In [2]:
# Bandingkan harga sekarang dengan snapshot sebelumnya (Lakehouse time travel)
snap = pd.read_sql('SELECT snapshot_id, committed_at, operation FROM lakehouse.distribusi."harga_produk$snapshots" ORDER BY committed_at DESC', engine)
print(snap.to_string(index=False))
prev_id = int(snap.iloc[int(os.environ.get("SNAP_INDEX", "1"))]["snapshot_id"])  # 1 = snapshot sebelumnya
lama = pd.read_sql(f"SELECT produk_id, harga_rupiah AS harga_lama FROM lakehouse.distribusi.harga_produk FOR VERSION AS OF {prev_id}", engine)
df = df.merge(lama, on="produk_id")
df["harga_lama"] = df["harga_lama"].astype(float)
df["rasio_perubahan"] = (df["harga_rupiah"] / df["harga_lama"]).round(3)
anomali = df[(df["rasio_perubahan"] > 10) | (df["rasio_perubahan"] < 0.1)]
print(len(anomali), "produk berubah harga lebih dari 10x sejak snapshot", prev_id)
anomali.groupby(["pemasok", "kategori", "berlaku_sejak"]).agg(
    jumlah=("sku", "count"), rasio_min=("rasio_perubahan", "min"), rasio_max=("rasio_perubahan", "max"))

        snapshot_id                     committed_at operation
4257562427355506277 2026-11-24 01:08:40.676000+00:00 overwrite
5109269415934974346 2026-11-24 01:08:29.486000+00:00    append
100 produk berubah harga lebih dari 10x sejak snapshot 5109269415934974346


,,,jumlah,rasio_min,rasio_max
pemasok,kategori,berlaku_sejak,,,
CV Tirta Jaya,Makanan Ringan,2026-11-23,100,1000.0,1000.0


In [3]:
anomali.sort_values("sku")[["sku", "kategori", "pemasok", "harga_lama", "harga_rupiah", "rasio_perubahan"]].head(5)

,sku,kategori,pemasok,harga_lama,harga_rupiah,rasio_perubahan
0,SKU-00001,Makanan Ringan,CV Tirta Jaya,11900.0,11900000.0,1000.0
1,SKU-00013,Makanan Ringan,CV Tirta Jaya,14900.0,14900000.0,1000.0
2,SKU-00025,Makanan Ringan,CV Tirta Jaya,18000.0,18000000.0,1000.0
3,SKU-00037,Makanan Ringan,CV Tirta Jaya,21000.0,21000000.0,1000.0
4,SKU-00049,Makanan Ringan,CV Tirta Jaya,24000.0,24000000.0,1000.0


In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4.5))
normal = df.drop(anomali.index)
ax.scatter(normal["produk_id"], normal["harga_rupiah"], s=6, color="#2563EB", label=f"Normal ({len(normal)})")
ax.scatter(anomali["produk_id"], anomali["harga_rupiah"], s=12, color="#DC2626", label=f"Berubah > 10x ({len(anomali)})")
ax.set_yscale("log"); ax.set_xlabel("produk_id"); ax.set_ylabel("harga_rupiah (skala log)")
ax.set_title(os.environ.get("CHART_TITLE", "lakehouse.distribusi.harga_produk"))
ax.legend(); fig.tight_layout(); fig.savefig(os.environ.get("CHART_OUT", "/home/jovyan/work/anomali.png"), dpi=130)
print("grafik disimpan")

grafik disimpan
